In [ ]:
class Character:
    def __init__(self, name, max_health, physical_damage, magic_damage, physical_armour,magic_armour, accuracy, pharmacology, moves=None):
        self.name = name
        self.maxValues = (max_health, physical_armour, magic_armour)
        self.current_health = max_health
        self.max_health = max_health
        self.physical_damage = physical_damage
        self.magic_damage = magic_damage
        self.physical_armour = physical_armour
        self.magic_armour = magic_armour
        self.accuracy = accuracy
        self.pharmacology = pharmacology
        # Which actions this character's AI can pick from: p = physical, m = magic, h = heal
        self.moves = moves if moves is not None else ['p', 'm']
        
    def take_magic_damage(self, dmgAmount):
        if self.magic_armour > 0: 
            self.magic_armour -= dmgAmount
            if self.magic_armour < 0:
                self.current_health -= abs(self.magic_armour) 
                self.magic_armour = 0 
        else:
            self.current_health -= dmgAmount
        return str(dmgAmount)

    def take_physical_damage(self, dmgAmount):
        if self.physical_armour > 0: 
            self.physical_armour -= dmgAmount
            if self.physical_armour < 0:
                self.current_health -= abs(self.physical_armour)
                self.physical_armour = 0 
        else:
            self.current_health -= dmgAmount
        return str(dmgAmount)
        
    def quick_heal(self, dmgAmount):
        self.current_health += dmgAmount
        if self.current_health >= self.max_health:
            self.current_health = self.max_health
        return str(dmgAmount)
    
    def take_damage(self,dmgAmount, dmgType):
        if dmgType == "physical":
            return self.name + " Took " + self.take_physical_damage(dmgAmount) + " " + dmgType + " damage"
        elif dmgType == "magic":
            return self.name + " Took " + self.take_magic_damage(dmgAmount) + " " + dmgType + " damage"
        elif dmgType == "heal":
            return self.name + " Recovered " + self.quick_heal(dmgAmount) + " " + dmgType + " health"
        else:
            return "Incorrect dmg type:" + dmgType

    def attack(self, characterToAttack, attackType):
        dmgAmount = 0
        if attackType == "physical":
            dmgAmount = self.physical_damage
        elif attackType == "magic":
            dmgAmount = self.magic_damage
        elif attackType == "heal":
            dmgAmount = self.pharmacology
        return characterToAttack.take_damage(dmgAmount,attackType)

    def physical_attack(self,characterToAttack):
        return self.attack(characterToAttack, "physical")

    def magic_attack(self,characterToAttack):
        return self.attack(characterToAttack, "magic")

    def heal(self,characterToAttack):
        return self.attack(characterToAttack, "heal")

    def __str__(self):
        return f"{self.name.upper()}\nHP: {self.current_health}\nPhys Arm: {self.physical_armour} | Mag Arm: {self.magic_armour}\nPhys Dmg: {self.physical_damage} | Mag Dmg: {self.magic_damage}"


In [ ]:
import tkinter as tk
from tkinter import scrolledtext
import random

# Stage counter. Only created if it doesn't exist yet, so re-running this cell won't reset your progress.
if 'monster_type' not in globals():
    monster_type = 0

# --- The Animated GUI Application ---
class BattleGUI:
    def __init__(self, root, player, enemy):
        self.root = root
        self.root.title("Animated RPG Battle")
        self.root.geometry("550x650") # Taller window to fit graphics
        
        self.player = player
        self.enemy = enemy
        self.battle_over = False  # guards against counting a win more than once

        # 1. Top Frame for Graphics (Canvas)
        self.canvas = tk.Canvas(root, width=500, height=180, bg="#e8e8e8", highlightthickness=2)
        self.canvas.pack(pady=15)
        
        # Draw Player Graphics (Blue)
        self.canvas.create_rectangle(50, 60, 110, 140, fill="royalblue", tags="player")
        self.canvas.create_text(80, 40, text="Player", fill="royalblue", font=("Courier", 12, "bold"), tags="player")
        
        # Draw Enemy Graphics (Red)
        self.canvas.create_rectangle(390, 60, 450, 140, fill="firebrick", tags="enemy")
        self.canvas.create_text(420, 40, text="Enemy", fill="firebrick", font=("Courier", 12, "bold"), tags="enemy")

        # 2. Frame for Character Stats
        self.stats_frame = tk.Frame(root)
        self.stats_frame.pack(fill=tk.X, padx=20)

        self.player_label = tk.Label(self.stats_frame, text=str(self.player), justify=tk.LEFT, font=("Courier", 10))
        self.player_label.pack(side=tk.LEFT)

        self.enemy_label = tk.Label(self.stats_frame, text=str(self.enemy), justify=tk.RIGHT, font=("Courier", 10))
        self.enemy_label.pack(side=tk.RIGHT)

        # 3. Frame for Combat Log
        self.log_area = scrolledtext.ScrolledText(root, wrap=tk.WORD, width=60, height=12, font=("Courier", 9))
        self.log_area.pack(pady=15)
        self.log_area.insert(tk.END, "Battle Commenced!\nChoose your attack below...\n\n")
        self.log_area.configure(state='disabled')

        # 4. Bottom Frame for Actions
        self.action_frame = tk.Frame(root)
        self.action_frame.pack(pady=5)

        self.btn_physical = tk.Button(self.action_frame, text="Physical Attack", command=self.do_physical_attack, width=15)
        self.btn_physical.pack(side=tk.LEFT, padx=10)

        self.btn_magic = tk.Button(self.action_frame, text="Magic Attack", command=self.do_magic_attack, width=15)
        self.btn_magic.pack(side=tk.RIGHT, padx=10)

        self.btn_heal = tk.Button(self.action_frame, text="Heal", command=self.do_heal, width=15) #new heal button here
        self.btn_heal.pack(side=tk.RIGHT, padx=15)


    # --- Animation Methods ---
    
    def animate_lunge(self, tag, distance, callback):
        """Simulates a physical attack by lunging the shape forward and back."""
        self.canvas.move(tag, distance, 0)
        self.root.update()
        # Wait 150ms, then snap back and trigger the callback
        self.root.after(150, lambda: self.finish_lunge(tag, distance, callback))

    def finish_lunge(self, tag, distance, callback):
        self.canvas.move(tag, -distance, 0)
        callback()

    def animate_magic(self, attacker_tag, callback):
        """Simulates a magic attack by shooting a yellow projectile."""
        if attacker_tag == "player":
            x1, y1, dx = 110, 100, 280
        else:
            x1, y1, dx = 390, 100, -280
            
        missile = self.canvas.create_oval(x1, y1-10, x1+20, y1+10, fill="gold")
        self.move_missile(missile, dx, 10, callback) # Move the distance in 10 steps

    def move_missile(self, missile, total_dx, steps_left, callback):
        if steps_left > 0:
            step_x = total_dx / 10
            self.canvas.move(missile, step_x, 0)
            self.root.after(30, lambda: self.move_missile(missile, total_dx, steps_left-1, callback))
        else:
            self.canvas.delete(missile)
            callback()

    # --- Game Logic Methods ---

    def write_log(self, text):
        self.log_area.configure(state='normal')
        self.log_area.insert(tk.END, text + "\n")
        self.log_area.see(tk.END) 
        self.log_area.configure(state='disabled')

    def refresh_stats(self):
        self.player_label.config(text=str(self.player))
        self.enemy_label.config(text=str(self.enemy))

    def check_winner(self):
        global monster_type  # we assign to it below, so it must be declared global
        if self.player.current_health <= 0 or self.enemy.current_health <= 0:
            self.disable_buttons()
            
            # Only announce the result and advance the stage once per battle
            if not self.battle_over:
                self.battle_over = True
                self.write_log("\n" + "="*30)
                if self.player.current_health > 0:
                    self.write_log(f"{self.player.name} has won the battle!!")
                    monster_type += 1
                elif self.enemy.current_health > 0:
                    self.write_log(f"{self.enemy.name} has won the battle!!")
                else:
                    self.write_log("It's a draw!")
            return True
        return False

    def disable_buttons(self):
        self.btn_physical.config(state=tk.DISABLED)
        self.btn_magic.config(state=tk.DISABLED)
        self.btn_heal.config(state=tk.DISABLED)

    def enable_buttons(self):
        if not self.check_winner():
            self.btn_physical.config(state=tk.NORMAL)
            self.btn_magic.config(state=tk.NORMAL)
            self.btn_heal.config(state=tk.NORMAL)

    # --- Turn Execution ---

    def do_physical_attack(self):
        self.disable_buttons()
        self.write_log("--- Player's Turn ---")
        # Trigger lunge animation, then apply damage
        self.animate_lunge("player", 50, lambda: self.resolve_player_turn("p"))

    def do_magic_attack(self):
        self.disable_buttons()
        self.write_log("--- Player's Turn ---")
        # Trigger magic animation, then apply damage
        self.animate_magic("player", lambda: self.resolve_player_turn("m"))
        
    def do_heal(self):
        self.disable_buttons()
        self.write_log("--- Player's Turn ---")
        # Trigger magic animation, then apply damage
        self.animate_magic("player", lambda: self.resolve_player_turn("h"))
        
    missing = False
    def hit(self, character):
        hit_rate = random.randint(1,10)
        missing = False
        if hit_rate > character.accuracy:
            missing = True
        return missing
        
    def resolve_player_turn(self, move_type):
        missing = self.hit(p1)
        if missing == True:
            result = "Missed!"
        else:
            extra_text = random.randint(1,5)
            if extra_text == 2:
                result = "Hit!\nWell Struck!"
            elif extra_text == 1:
                result = "Hit!\nClean Though!"
            elif extra_text == 3:
                result = "Hit!\nAnother Good Strike!"
            elif extra_text == 4:
                result = "Hit!\nThey'll Regreat Crossing You!"
            else:
                result = "Hit!\nDid you expect anything else?"
            if move_type == "m":
                p1.magic_attack(p2)
            elif move_type == "h":
                p1.heal(p1)
            else:
                p1.physical_attack(p2)
        self.write_log(f"-> {result}")
        self.refresh_stats()
        
        if self.check_winner(): return
        
        self.write_log("\n--- Enemy's Turn ---")
        # Pause for half a second so the user can read what happened before enemy attacks
        self.root.after(600, self.enemy_turn)

    def enemy_turn(self):
        enemy_move = random.choice(self.enemy.moves)
        self.write_log(f"Enemy selected: {enemy_move}")
        
        if enemy_move == "p":
            self.animate_lunge("enemy", -50, lambda: self.resolve_enemy_turn("p"))
        elif enemy_move == "h":
            self.animate_magic("enemy", lambda: self.resolve_enemy_turn("h"))
        else:
            self.animate_magic("enemy", lambda: self.resolve_enemy_turn("m"))
            
    def resolve_enemy_turn(self, move_type):
        missing = self.hit(p2)
        if missing == True:
            result = "Missed"
        else:
            extra_text = random.randint(1,5)
            if extra_text == 1:
                result = "Hit!\nHow Quickly the Tides Change!"
            elif extra_text == 2:
                result = "Hit!\nYou're Closer to the End!"
            else:
                result = "Hit!"
            if move_type == "m":
                p2.magic_attack(p1)
            elif move_type == "h":
                p2.heal(p2)
            else:
                p2.physical_attack(p1)

        self.write_log(f"-> {result}\n")
        self.refresh_stats()
        self.enable_buttons() 

In [15]:
p_c = 0
player_class = input("Pick a class\nK = Kinght, M = Mage, H = Highwayman\n")
player_class = player_class.lower()
if "k" in player_class:
    p_c = 1
elif "m" in player_class:
    p_c = 2
elif "h" in player_class:
    p_c = 3
elif "trapper" in player_class:
    p_c = 4
elif "doppelganger" in player_class:
    p_c = 5
else:
    print("Class not reconized")

Pick a class
K = Kinght, M = Mage, H = Highwayman
 K


In [ ]:
# --- Main Application Execution ---
#Start by creating some characters and assigning them to p1 & p2!
global p1 
global p2

if p_c == 1:
    p1 = Character("Knight", 100, 12, 12, 25, 20, 6, 24)
elif p_c == 2:
    p1 = Character("Mage", 100, 7, 23, 8, 35, 6, 46)
elif p_c == 3:
    p1 = Character("Highwayman", 90, 30, 5, 15, 10, 6, 10)
elif p_c == 4:
    p1 = Character("Trapper", 60, 8, 8, 10, 10, 10, 12)
elif p_c == 5:
    if monster_type == 0:
        p1 = Character("The Doppelganger", 50, 7, 3, 10, 2, 4, 3)
    elif monster_type == 1:
        p1 = Character("The Doppelganger", 60, 6, 16, 12, 20, 4, 16)
    elif monster_type == 2:
        p1 = Character("The Doppelganger", 60, 8, 8, 10, 10, 10, 12)
    elif monster_type == 3:
        p1 = Character("The Doppelganger", 30, 5, 5, 50, 30, 7, 10)
    elif monster_type == 4:
        p1 = Character("The Doppelganger", 50, 20, 0, 20, 20, 6, 10)
    elif monster_type == 5:
        p1 = Character("The True Doppelganger", 250, 30, 40, 30, 40, 8, 40)
    elif monster_type == 6:
        p1 = Character("The Doppelganger", 120, 20, 20, 30, 20, 7, 30)
    else:
        p1 = Character("The True Doppelganger", 250, 30, 40, 30, 40, 8, 40)

    
if monster_type == 0:
    p2 = Character("Skeever", 50, 7, 3, 10, 2, 4, 3, moves=['p', 'm'])
elif monster_type == 1:
    p2 = Character("Mage", 60, 6, 16, 12, 20, 4, 16, moves=['m'])
elif monster_type == 2:
    p2 = Character("Trapper", 60, 8, 8, 10, 10, 10, 12, moves=['p', 'm', 'h'])
elif monster_type == 3:
    p2 = Character("Ghost", 30, 5, 5, 50, 30, 7, 10, moves=['p', 'm', 'h'])
elif monster_type == 4:
    p2 = Character("Runaway", 50, 20, 0, 20, 20, 6, 10, moves=['p', 'p', 'h'])
elif monster_type == 5:
    if p_c == 1:
        p2 = Character("The Doppelganger", 100, 12, 12, 25, 20, 6, 24, moves=['p', 'm', 'h'])
    elif p_c == 2:
        p2 = Character("The Doppelganger", 100, 7, 23, 8, 35, 6, 46, moves=['p', 'm', 'h'])
    elif p_c == 3:
        p2 = Character("The Doppelganger", 90, 30, 5, 15, 10, 6, 10, moves=['p', 'm', 'h'])
    elif p_c == 4:
        p2 = Character("The Doppelganger", 60, 8, 8, 10, 10, 10, 12, moves=['p', 'm', 'h'])
    elif p_c == 5:
        p2 = Character("The True Doppelganger", 250, 30, 40, 30, 40, 8, 40, moves=['p', 'm', 'h'])
elif monster_type == 6:
    p2 = Character("Mockingbird", 120, 20, 20, 30, 20, 7, 30, moves=['p', 'm'])
elif monster_type == 7:
    p2 = Character("Maskless Trapper", 130, 12, 12, 0, 0, 10, 20, moves=['p', 'm', 'h'])
elif monster_type == 8:
    p2 = Character("Enstraged Cultist", 60, 10, 26, 40, 32, 4, 10, moves=['p', 'm'])
elif monster_type == 9:
    p2 = Character("Faceless Knight", 200, 12, 12, 25, 20, 6, 24, moves=['p', 'm'])
elif monster_type == 10:
    if p_c == 1:
        p2 = Character("The Doppelganger", 100, 12, 12, 25, 20, 6, 24, moves=['p', 'm'])
    elif p_c == 2:
        p2 = Character("The Doppelganger", 100, 7, 23, 8, 35, 6, 46, moves=['p', 'm'])
    elif p_c == 3:
        p2 = Character("The Doppelganger", 90, 30, 5, 15, 10, 6, 10, moves=['p', 'm'])
    elif p_c == 4:
        p2 = Character("The Doppelganger", 60, 8, 8, 10, 10, 10, 12, moves=['p', 'm'])
    elif p_c == 5:
        p2 = Character("The True Doppelganger", 250, 30, 40, 30, 40, 8, 40, moves=['p', 'm'])
elif monster_type == 11:
    p2 = Character("noTHING", 320, 20, 23, 30, 40, 8, 40, moves=['p', 'm'])
elif monster_type > 11:
    p2 = Character("Shifter", 135, 23, 28, 10, 3, 3, 348367, moves=['p', 'm']) #stats were randomly generated via random number generater. Shifter is a Being of random
root = tk.Tk()
app = BattleGUI(root, p1, p2)
root.mainloop()

In [ ]:
# Run this cell to reset which stage you're at, then re-run the class-select and battle cells
monster_type = 0
